In [11]:
%pwd

'/Users/utkuayten/Desktop/soffritto-org'

In [2]:
import numpy as np
import torch
import torch.nn.functional as F

def load_first_npz_array(npz_path: str) -> np.ndarray:
    z = np.load(npz_path)
    return z[z.files[8]]  # no guessing/optional logic beyond "first array"

def kl_pq_mean_and_per_sample(p_true: torch.Tensor, q_pred: torch.Tensor):
    """
    KL(P||Q) where P=true labels (prob), Q=predictions (prob).
    Shapes: (N, K)
    Returns: (kl_per_sample: (N,), kl_mean: scalar)
    """
    # input must be log-probs, target must be probs
    kl_elem = F.kl_div(q_pred.log(), p_true, reduction="none")  # (N, K)
    kl_per_sample = kl_elem.sum(dim=1)                          # (N,)
    kl_mean = kl_per_sample.mean()                              # scalar
    return kl_per_sample, kl_mean


cell_lines = ['H1']
for cell in cell_lines:
    pred = torch.from_numpy(np.load(f"GAT/predictions/{cell}_chr9_pred_intra_cell_line_GAT.npy")).to(torch.float64)
    labels = torch.from_numpy(load_first_npz_array(f"Soffritto/data/{cell}_labels.npz")).to(torch.float64)
    assert pred.shape == labels.shape, f"Shape mismatch: pred{pred.shape} vs labels{labels.shape}"

    kl_each, kl_avg = kl_pq_mean_and_per_sample(labels, pred)
    print(f"For cell {cell} KL loss: {kl_avg:.5f}")
    print("First 5 KL values:", kl_each[:5],"\n" )

For cell H1 KL loss: 0.05800
First 5 KL values: tensor([0.1761, 0.1123, 0.0899, 0.0398, 0.0317], dtype=torch.float64) 

